# Pre-Loan Eligibility Prediction

This notebook implements pre-loan eligibility assessment using applicant-related financial features.

## Define Pre-Loan Features

In [1]:
preloan_features = [
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "purpose",
    "dti",
    "fico_range_low",
    "fico_range_high"
]
# Features available before loan approval

## Define Target Variable

In [2]:
target = "loan_default"

## Load Dataset

In [3]:
import pandas as pd

df = pd.read_csv("loan_default_final_ready.csv")# Load cleaned Lending Club dataset

X = df[preloan_features]
y = df[target]

## Train-Test Split

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
) # Preserve class distribution using stratified sampling

## Build Preprocessing Pipeline

In [5]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns # Scale numerical variables
cat_cols = X_train.select_dtypes(include=["object", "string"]).columns # Encode categorical variables

preloan_preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)

## Logistic Regression Model

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Train Logistic Regression eligibility model
lr_model = Pipeline(steps=[
    ("preprocess", preloan_preprocess),
    ("model", LogisticRegression(max_iter=1000))
])

lr_model.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti',
       'fico_range_low', 'fico_range_high'],
      dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['term', 'grade', 'sub_grade', 'emp_length', 'home_ownership',
       'verification_status', 'purpose'],
      dtype='object'))])),
                ('model', LogisticRegression(max_iter=1000))])

## Evaluate Logistic Regression

In [7]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred_lr = lr_model.predict(X_test)
y_prob_lr = lr_model.predict_proba(X_test)[:, 1]

print("Logistic Regression")
print(classification_report(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_lr))

Logistic Regression
              precision    recall  f1-score   support

           0       0.58      0.59      0.59      9492
           1       0.62      0.62      0.62     10508

    accuracy                           0.60     20000
   macro avg       0.60      0.60      0.60     20000
weighted avg       0.61      0.60      0.61     20000

ROC-AUC: 0.6468863207146891


## XGBoost Model

In [8]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

xgb_model = Pipeline(steps=[
    ("preprocess", preloan_preprocess),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    ))
])

xgb_model.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti',
       'fico_range_low', 'fico_range_high'],
      dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['term', 'grade', 'sub_grade', 'emp_length', 'home_ownership',
       'verification_status', 'purpose'],
      dtype...
                               feature_types=None, gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=5, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=300, n_jobs=None,
                               num_parallel_tree=None, random_state=42, ...))])

## Evaluate XGBoost

In [9]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("XGBoost – Pre-Loan Eligibility")
print(classification_report(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_xgb))

XGBoost – Pre-Loan Eligibility
              precision    recall  f1-score   support

           0       0.72      0.68      0.70      9492
           1       0.72      0.76      0.74     10508

    accuracy                           0.72     20000
   macro avg       0.72      0.72      0.72     20000
weighted avg       0.72      0.72      0.72     20000

ROC-AUC: 0.790144744132498


## Stacking Ensemble

In [10]:
from sklearn.ensemble import StackingClassifier

stacking_model = Pipeline(steps=[
    ("preprocess", preloan_preprocess),
    ("model", StackingClassifier(
        estimators=[
            ("lr", LogisticRegression(max_iter=1000)),
            ("xgb", XGBClassifier(
                n_estimators=150,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                eval_metric="logloss",
                random_state=42
            ))
        ],
        final_estimator=LogisticRegression(),
        passthrough=False
    ))
])

stacking_model.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti',
       'fico_range_low', 'fico_range_high'],
      dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['term', 'grade', 'sub_grade', 'emp_length', 'home_ownership',
       'verification_status', 'purpose'],
      dtype...
                                                               interaction_constraints=None,
                                                               learning_rate=0.05,
                                                               max_bin=None,
                                                               max_cat_threshold=None,
                                                               max_cat_to_onehot=None,
                                                               max_delta_step=None,
                                                               max_depth=4,
                                                               max_leaves=None,
                                                               min_child_weight=None,
                                                               missing=nan,
                                                               monotone_constraints=None,
                                                               multi_strategy=None,
                                                               n_estimators=150,
                                                               n_jobs=None,
                                                               num_parallel_tree=None,
                                                               random_state=42, ...))],
                                    final_estimator=LogisticRegression()))])

## Evaluate Stacking Ensemble

In [11]:
y_pred_stack = stacking_model.predict(X_test)
y_prob_stack = stacking_model.predict_proba(X_test)[:, 1]

print("Stacking Ensemble")
print(classification_report(y_test, y_pred_stack))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_stack))

Stacking Ensemble
              precision    recall  f1-score   support

           0       0.67      0.66      0.66      9492
           1       0.70      0.71      0.70     10508

    accuracy                           0.68     20000
   macro avg       0.68      0.68      0.68     20000
weighted avg       0.68      0.68      0.68     20000

ROC-AUC: 0.7444279304945515


## Save Final Pre-Loan Model

In [12]:
import os
import joblib

# 1️ Create directory for pre-loan models
os.makedirs("models/preloan", exist_ok=True)

# 2️ Save ONLY the pre-loan Logistic Regression pipeline
joblib.dump(
    lr_model,
    "models/preloan/eligibility_lr.pkl"
)
print("Pre-loan eligibility model saved successfully!")

Pre-loan eligibility model saved successfully!
